# Efficient MITM Detection Pipeline
### CNN + LSTM Hybrid Architecture with SMOTE Balancing

This notebook implements an optimized 13-step pipeline for detecting MITM attacks. It handles data merging, feature engineering, recursive feature selection, and hybrid deep learning training.


In [ ]:
import os, glob, joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, MaxPooling1D, Dropout, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Configuration for efficiency
os.makedirs('model', exist_ok=True)
os.makedirs('plots', exist_ok=True)
np.random.seed(42)
tf.random.set_seed(42)
print("Environment Ready.")


## 1. Load and Merge All 5 Dataests


In [ ]:
all_files = glob.glob("dataset/*.csv")
df_list = []
for f in all_files:
    temp = pd.read_csv(f, low_memory=False)
    # Handle potential Label naming variation
    if "Label" not in temp.columns:
        lbl_col = [c for c in temp.columns if c.lower() == "label"][0]
        temp.rename(columns={lbl_col: "Label"}, inplace=True)
    df_list.append(temp)
    print(f"Loaded {os.path.basename(f)}: {temp.shape}")

df = pd.concat(df_list, axis=0, ignore_index=True)
print(f"
Total Merged Data: {df.shape}")
df["Label"].value_counts().plot(kind="bar", title="Initial Label Distribution")
plt.show()


## 2. Column Analysis & Dropping


In [ ]:
cols_to_drop = ["id", "expiration_id", "src_ip", "src_mac", "src_oui", "dst_ip", "dst_mac", "dst_oui", 
                "vlan_id", "tunnel_id", "bidirectional_first_seen_ms", "bidirectional_last_seen_ms",
                "src2dst_first_seen_ms", "src2dst_last_seen_ms", "dst2src_first_seen_ms", "dst2src_last_seen_ms",
                "user_agent", "content_type", "requested_server_name", "client_fingerprint", "server_fingerprint",
                "application_name", "application_category_name", "application_is_guessed", "application_confidence"]

# Drop only existing columns
existing_drops = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=existing_drops, inplace=True)
print(f"Dropped {len(existing_drops)} columns. Remaining: {df.shape[1]}")


## 3. Label Encoding (Binary)


In [ ]:
def encode_label(val):
    v = str(val).lower()
    if v in ["normal", "benign", "0", "0.0"]: return 0
    return 1 # MITM, Attack, ARP, etc.

df["Label"] = df["Label"].apply(encode_label)
print("Class distribution:", df["Label"].value_counts())


## 4. Handle Missing Values


In [ ]:
df.fillna(0, inplace=True)
df.replace([np.inf, -np.inf], 0, inplace=True)
print("Missing and infinite values resolved.")


## 5. MITM-Specific Feature Engineering


In [ ]:
df["packet_asymmetry"] = abs(df["src2dst_packets"] - df["dst2src_packets"]) / (df["bidirectional_packets"] + 1)
df["byte_asymmetry"] = abs(df["src2dst_bytes"] - df["dst2src_bytes"]) / (df["bidirectional_bytes"] + 1)
df["bytes_per_packet"] = df["bidirectional_bytes"] / (df["bidirectional_packets"] + 1)
df["src2dst_bpp"] = df["src2dst_bytes"] / (df["src2dst_packets"] + 1)
df["dst2src_bpp"] = df["dst2src_bytes"] / (df["dst2src_packets"] + 1)
df["duration_ratio"] = df["src2dst_duration_ms"] / (df["dst2src_duration_ms"] + 1)
df["syn_ratio"] = df["bidirectional_syn_packets"] / (df["bidirectional_packets"] + 1)
df["rst_ratio"] = df["bidirectional_rst_packets"] / (df["bidirectional_packets"] + 1)
df["piat_variance_ratio"] = df["bidirectional_stddev_piat_ms"] / (df["bidirectional_mean_piat_ms"] + 1)
df["ps_variance_ratio"] = df["bidirectional_stddev_ps"] / (df["bidirectional_mean_ps"] + 1)
print("Engineered 10 new features.")


## 6. Optimized Feature Selection (RF-RFE)


In [ ]:
X_fs = df.drop("Label", axis=1)
y_fs = df["Label"]

# Sample for efficiency (50k rows)
sample_size = min(50000, len(df))
X_samp, _, y_samp, _ = train_test_split(X_fs, y_fs, train_size=sample_size, stratify=y_fs, random_state=42)

print("Selecting top 25 features...")
rf = RandomForestClassifier(n_jobs=-1, n_estimators=50, random_state=42)
# step=5 for speed (removes 5 features per iteration)
rfe = RFE(estimator=rf, n_features_to_select=25, step=5) 
rfe.fit(X_samp, y_samp)

selected_features = X_fs.columns[rfe.support_].tolist()
joblib.dump(selected_features, "model/selected_features.pkl")

# Plot Feature Importance of selected features
importances = rfe.estimator_.feature_importances_
feat_imp = pd.Series(importances, index=selected_features).sort_values(ascending=False)
feat_imp.head(20).plot(kind="barh", figsize=(10,6), title="Top 20 Selected Features")
plt.gca().invert_yaxis()
plt.show()

df = df[selected_features + ["Label"]]


## 7-9. Split, Normalize, and SMOTE


In [ ]:
X = df.drop("Label", axis=1)
y = df["Label"]

# 70/10/20 Split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.66, stratify=y_temp, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, "model/scaler.pkl")

# SMOTE
print("Before SMOTE:", y_train.value_counts().to_dict())
smote = SMOTE(k_neighbors=5, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print("After SMOTE:", y_train_res.value_counts().to_dict())


## 10. CNN+LSTM Architecture


In [ ]:
n_feat = X_train_res.shape[1]
X_train_c = X_train_res.reshape(-1, n_feat, 1)
X_val_c = X_val_scaled.reshape(-1, n_feat, 1)
X_test_c = X_test_scaled.reshape(-1, n_feat, 1)

model = Sequential([
    Conv1D(64, 3, activation="relu", padding="same", input_shape=(n_feat, 1)),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.2),
    Conv1D(128, 3, activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer=Adam(0.001), loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
model.summary()


## 11. Optimized Training


In [ ]:
callbacks = [
    EarlyStopping(monitor="val_auc", patience=5, restore_best_weights=True, mode="max"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
]

history = model.fit(
    X_train_c, y_train_res, 
    validation_data=(X_val_c, y_val),
    epochs=20, batch_size=256, callbacks=callbacks, verbose=1
)


## 12. Performance Evaluation


In [ ]:
# Predictions (Threshold 0.7)
y_pred_prob = model.predict(X_test_c)
y_pred = (y_pred_prob > 0.7).astype(int)

print("
Classification Report:")
print(classification_report(y_test, y_pred))

# Plot Training History
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Val")
plt.title("Accuracy")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.title("Loss")
plt.legend()
plt.show()

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "MITM"], yticklabels=["Normal", "MITM"])
plt.title("Confusion Matrix")
plt.show()


## 13. Save Final Model


In [ ]:
model.save("model/mitm_model.h5")
print("Pipeline Complete. Model saved to model/mitm_model.h5")
